# CO₂ Storage Capacity Assessment

This notebook runs a probabilistic static storage-capacity assessment using:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The example values represent the Rødby Bunter Sandstone assessment.

In [ ]:
# Install the latest package and plotting tools from GitHub.
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")

## Editable inputs

Enter minimum, most likely, and maximum values. Fractions such as porosity must be decimals: `0.23` means 23%. The Rødby values come from GEUS Report 2024/18, Table 8.4.1.

**Source corrections:** Table 8.2.1 prints the maximum GRV as 23.9 km³, while Table 8.4.1 gives 33.85 km³; the latter agrees with the stated ±20% method around the 28.21 km³ mode. Table 8.4.1 also prints the maximum CO₂ density as 764.0 kg/m³, but Section 8.2.4 says the maximum is 10% above the 603.6 kg/m³ mode. The internally consistent values, 33.85 km³ and 663.96 kg/m³, are used below because they reproduce the report's capacity results.

In [ ]:
site_name = "Rødby – Bunter Sandstone"
iterations = 100_000
random_seed = 42

#                       minimum, most likely, maximum
grv_km3 =               (22.57, 28.21, 33.85)
net_to_gross =          (0.20,  0.25,  0.30)
porosity =              (0.184, 0.23,  0.276)
co2_density_kg_m3 =     (573.4, 603.6, 663.96)  # Table prints 764.0; Section 8.2.4 implies 663.96
storage_efficiency =    (0.05,  0.10,  0.20)

## Input parameter table

This table is generated from the editable values above, so it updates automatically when you change an input. `Description / interpretation`, `Input type`, and `Source / basis` consolidate the meaning and provenance of every Rødby parameter in one place.

In [ ]:
input_table = pd.DataFrame([
    ["Gross rock volume (GRV)", "km³", "PERT", *grv_km3, "Seismic-derived estimate", "Seismic-mapped Bunter Sandstone reservoir volume within the Rødby structural closure. The range comes from the structural volumetric model; 33.85 km³ is the value consistent with Table 8.4.1 and the stated ±20% method.", "GEUS Report 2024/18, Sections 8.1–8.2.1 and Table 8.4.1"],
    ["Net-to-gross (N/G)", "fraction", "PERT", *net_to_gross, "Geological / well-derived estimate", "Fraction of the gross Bunter Sandstone interval interpreted as effective reservoir sandstone, based on Rødby-1, Rødby-2 and surrounding wells.", "GEUS Report 2024/18, Section 8.2.2 and Table 8.4.1"],
    ["Porosity (φ)", "fraction", "PERT", *porosity, "Petrophysical estimate", "Fraction of reservoir bulk volume occupied by pore space. The 23% mode comes from well-based petrophysical interpretation, with approximately ±20% uncertainty for spatial and vertical variability.", "Rødby-1, Rødby-2 and surrounding wells; GEUS Report 2024/18, Sections 7.1 and 8.2.3"],
    ["In-situ CO₂ density", "kg/m³", "PERT", *co2_density_kg_m3, "Thermodynamic estimate", "Calculated CO₂ density at reservoir conditions, not a direct well measurement. GEUS applies about −5%/+10% around 603.6 kg/m³; the maximum is therefore 603.6 × 1.10 = 663.96 kg/m³.", "GEUS Report 2024/18, Section 8.2.4; CO₂ properties based on Span & Wagner (1996)"],
    ["Storage efficiency", "fraction", "PERT", *storage_efficiency, "Literature-informed, site-specific assumption", "Assumed fraction of available pore volume effectively occupied by stored CO₂. It is a screening assumption, not a direct Rødby measurement. The 10% mode reflects generally good Bunter Sandstone properties; revise it when dynamic simulation or more site-specific data become available.", "GEUS Report 2024/18, Sections 5.5 and 8.3; Goodman et al. (2011), Gorecki et al. (2009), Wang et al. (2013)"],
], columns=["Parameter", "Unit", "Distribution", "Minimum", "Mode", "Maximum", "Input type", "Description / interpretation", "Source / basis"])
input_table

## Rødby input-data provenance

The probabilistic inputs are based on Abramovitz et al. (2024), GEUS Report 2024/18, and use independent PERT distributions in the static volumetric equation:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

The parameter descriptions, input classifications, uncertainty rationale, and source basis are consolidated in the **Input parameter table** above.

### References

- Abramovitz, T., Vosgerau, H., Gregersen, U., et al. (2024). *CCS2022-2024 WP1: The Rødby structure – Seismic data and interpretation to mature potential geological storage of CO₂*. GEUS Report 2024/18. [https://doi.org/10.22008/gpub/34739](https://doi.org/10.22008/gpub/34739)
- Goodman, A., Hakala, J. A., Bromhal, G., Deel, D., Rodosta, T., Frailey, S., et al. (2011). *U.S. DOE methodology for the development of geologic storage potential for carbon dioxide at the national and regional scale*. *International Journal of Greenhouse Gas Control, 5*(4), 952–965. [https://doi.org/10.1016/j.ijggc.2011.03.010](https://doi.org/10.1016/j.ijggc.2011.03.010)
- Gorecki, C. D., Sorensen, J. A., Bremer, J. M., et al. (2009). *Development of Storage Coefficients for Carbon Dioxide Storage in Deep Saline Formations*. IEA Greenhouse Gas R&D Programme.
- Span, R. & Wagner, W. (1996). *A New Equation of State for Carbon Dioxide Covering the Fluid Region from the Triple-Point Temperature to 1100 K at Pressures up to 800 MPa*. *Journal of Physical and Chemical Reference Data, 25*(6), 1509–1596. [https://doi.org/10.1063/1.555991](https://doi.org/10.1063/1.555991)
- Wang, Y., Zhang, K. & Wu, N. (2013). *Numerical Investigation of the Storage Efficiency Factor for CO₂ Geological Sequestration in Saline Formations*. *Energy Procedia, 37*, 5267–5274. [https://doi.org/10.1016/j.egypro.2013.06.443](https://doi.org/10.1016/j.egypro.2013.06.443)

In [ ]:
site = StorageSite(
    name=site_name,
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)

result = simulate(site, iterations=iterations, seed=random_seed)
summary = result.summary()
report_values = {"p90_mt": 68.83, "p50_mt": 103.90, "p10_mt": 148.78, "mean_mt": 107.04}
result_rows = [("P90 (conservative)", "p90_mt"), ("P50 (median)", "p50_mt"), ("P10 (upside)", "p10_mt"), ("Mean", "mean_mt")]
comparison = pd.DataFrame({
    "Notebook (Mt CO₂)": [summary[key] for _, key in result_rows],
    "GEUS Table 8.5.1 (Mt CO₂)": [report_values[key] for _, key in result_rows],
}, index=[label for label, _ in result_rows])
comparison["Difference (Mt CO₂)"] = comparison["Notebook (Mt CO₂)"] - comparison["GEUS Table 8.5.1 (Mt CO₂)"]
comparison.round(2)

The notebook and GEUS values should be very close, but not numerically identical. Both use the same static volumetric equation and independent PERT inputs. Small differences are expected because the report does not state its number of Monte Carlo iterations, random seed, or exact software implementation of PERT.

## Input uncertainty distributions

In [ ]:
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
input_ranges = {
    "grv_km3": grv_km3,
    "net_to_gross": net_to_gross,
    "porosity": porosity,
    "co2_density_kg_m3": co2_density_kg_m3,
    "storage_efficiency": storage_efficiency,
}
line_styles = [
    ("Minimum", "#1f77b4", ":"),
    ("Mode", "#2ca02c", "--"),
    ("Mean", "#ff7f0e", "-"),
    ("Maximum", "#d62728", ":"),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, values) in zip(axes.flat, result.inputs.items()):
    minimum, mode, maximum = input_ranges[name]
    mean = float(np.mean(values))
    reference_values = [minimum, mode, mean, maximum]
    ax.hist(values, bins=45, color="#b9d7f0", edgecolor="white", alpha=0.9)
    for (line_name, color, linestyle), reference in zip(line_styles, reference_values):
        ax.axvline(
            reference,
            color=color,
            linestyle=linestyle,
            linewidth=1.6,
            label=f"{line_name}: {reference:.4g}",
        )
    ax.set_title(labels[name])
    ax.set_ylabel("Simulations")
    ax.legend(fontsize=8, frameon=True, loc="upper right")
axes.flat[-1].axis("off")
fig.suptitle(f"{site_name} – input uncertainty distributions and values used", fontsize=15)
fig.tight_layout()
plt.show()


## Storage-capacity probability distribution

In [ ]:
fig, ax = result.plot_pdf()
plt.show()

## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.

In [ ]:
fig, ax = result.plot_exceedance()
plt.show()

## Linear capacity confidence ranges

The colored bar summarizes conservative, central, and upside capacity ranges. These are probabilistic static capacity estimates, not booked reserves.

In [ ]:
fig, ax = result.plot_capacity_ranges()
plt.show()

## Sensitivity tornado chart (Spearman rank)

Longer bars identify the assumptions with the strongest association with calculated capacity. The number printed on each bar is its Spearman rank-correlation coefficient; it is not a percentage contribution to capacity or uncertainty.

In [ ]:
fig, ax = result.plot_sensitivity()
fig.set_size_inches(10, 6)

readable_tick_labels = [labels.get(tick.get_text(), tick.get_text()) for tick in ax.get_yticklabels()]
ax.set_yticks(ax.get_yticks(), labels=readable_tick_labels)

for bar in ax.patches:
    value = float(bar.get_width())
    y = bar.get_y() + bar.get_height() / 2
    if abs(value) >= 0.14:
        inset = 0.025 if value >= 0 else -0.025
        ax.text(
            value - inset,
            y,
            f"{value:.2f}",
            ha="right" if value >= 0 else "left",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    else:
        offset = 5 if value >= 0 else -5
        ax.annotate(
            f"{value:.2f}",
            (value, y),
            xytext=(offset, 0),
            textcoords="offset points",
            ha="left" if value >= 0 else "right",
            va="center",
            color="#1f1f1f",
            fontsize=9,
            fontweight="bold",
        )

plt.show()


## Important limitation

This is a static volumetric screening assessment. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation, or economics.